In [1]:
import pandas as pd
import numpy as np
from pyMyriad import *

In [2]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "A": np.random.normal(0, 1, 1000),
  "B": np.random.normal(0, 10, 1000)
})

df.head()

for i, col in enumerate(df.columns):
    print(i, col)

0 id
1 Gender
2 Age
3 Income
4 Country
5 A
6 B


# Basic usage
The analysis tree is constructed by successive split and analyis adding nodes to the tree.
- `split_*` methods add split nodes, dividing or filtering the dataframe.
- `analyze_*` methods add an analysis node that will perform the analysis.

Split and analysis can be specified either as sting of functions.
Note: use `df` to refer to the dataframe available at the node.

The resulting analyis tree can be printed to visualize the analysis plan

In [3]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age > 40", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun)

print(atree)

Analysis Tree
  └- Split Node df.Gender: [df.Gender]
    └- Split Node Benin Y/N: [<function>]
      └- Split Node Age > 40: [age40: <function> -- age60: <function>]
        └- Analysis Node: 
            m: <function>
            n: <function>



The `run` method applies the analysis plan on a dataframe and returns a `DataTree`.

In [4]:
res = atree.run(df)
print(res)

Data Tree
  Split: df.Gender
    └- F
      Split: Benin Y/N
        └- False
          Split: Age > 40
            └- age40
                analysis: 
                └- m: 49567.51750644849
                └- n: 15248.68388993369
            └- age60
                analysis: 
                └- m: 48573.163158597716
                └- n: 17932.834511334153
        └- True
          Split: Age > 40
            └- age40
                analysis: 
                └- m: 51549.03235906791
                └- n: 15382.20729670539
            └- age60
                analysis: 
                └- m: 53242.501731397264
                └- n: 15936.002074205217
    └- M
      Split: Benin Y/N
        └- False
          Split: Age > 40
            └- age40
                analysis: 
                └- m: 48280.63036474299
                └- n: 14670.786165229021
            └- age60
                analysis: 
                └- m: 48294.682783294906
                └- n: 11325.740992012908
    

In [10]:


atree = AnalysisTree(id = "id")\
  .split_by(label = "first split", expr = "df.Gender")\
  .split_by(label = "second split", expr = "df.Country == 'Benin'")\
  .split_at_by(path = [], label = "xalso first split", expr = benin_fun)\
  .analyze_by(m = mfun, n = nfun)\
  .analyze_by_at(path = [], label = "root analysis", m = mfun, n = nfun)

print(atree)

res = atree.run(df)
print(res)

Analysis Tree
  └- Split Node first split: [df.Gender]
    └- Split Node second split: [df.Country == 'Benin']
      └- Analysis Node: 
          m: <function>
          n: <function>
  └- Split Node xalso first split: [<function>]
    └- Analysis Node: 
        m: <function>
        n: <function>
  └- Analysis Node: root analysis
      m: <function>
      n: <function>

Data Tree
  Split: first split
    └- F
      Split: second split
        └- False
            analysis: 
            └- m: 49062.75040449646
            └- n: 15367.203778709894
        └- True
            analysis: 
            └- m: 52155.247009503844
            └- n: 14348.039973032479
    └- M
      Split: second split
        └- False
            analysis: 
            └- m: 47670.17490398412
            └- n: 14725.472914363218
        └- True
            analysis: 
            └- m: 49060.94555672068
            └- n: 13620.272937819987
  Split: xalso first split
    └- False
        analysis: 
        └- m: 4

In [11]:
from pyMyriad.plots import forest_plot
ff = forest_plot(res, "m", "n")
display(ff)

[   type split   lvl    path path_pivot pivot_split pivot_lvl  depth label  \
0  root  None  None  [root]     [root]          []        []      0  None   

  summary  
0    None  ,         type         split    lvl  \
0      split   first split   None   
1      level          None      F   
2      split  second split   None   
3      level          None  False   
4   analysis          None   None   
5      level          None   True   
6   analysis          None   None   
7      level          None      M   
8      split  second split   None   
9      level          None  False   
10  analysis          None   None   
11     level          None   True   
12  analysis          None   None   

                                                 path  \
0                                 [root, first split]   
1                              [root, first split, F]   
2                [root, first split, F, second split]   
3         [root, first split, F, second split, False]   
4   [root, firs

statistics,y,split,type,path_pivot,pivot_lvl,pivot_split,label,y_label,NaN,m,n
0,1.0,NaN,analysis,root >> analysis,,,root analysis,root,NaN,49484.669153,14639.714985
1,2.0,first split,split,root >> first split,,,,first split,None,NaN,NaN
2,3.0,NaN,level,root >> first split >> F,,,,F,None,NaN,NaN
3,4.0,second split,split,root >> first split >> F >> second split,,,,second split,None,NaN,NaN
4,5.0,NaN,analysis,root >> first split >> F >> second split >> Fa...,,,,False,NaN,49062.750404,15367.203779
5,6.0,NaN,analysis,root >> first split >> F >> second split >> Tr...,,,,True,NaN,52155.24701,14348.039973
6,7.0,NaN,level,root >> first split >> M,,,,M,None,NaN,NaN
7,8.0,second split,split,root >> first split >> M >> second split,,,,second split,None,NaN,NaN
8,9.0,NaN,analysis,root >> first split >> M >> second split >> Fa...,,,,False,NaN,47670.174904,14725.472914
9,10.0,NaN,analysis,root >> first split >> M >> second split >> Tr...,,,,True,NaN,49060.945557,13620.272938
